# 01 – Download arXiv abstracts and build the dataset table

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilaria-cesa/Txt_lab/blob/main/notebooks/01_download_and_build_dataset.ipynb)

Work in progress. This notebook runs the whole data pipeline:

1. downloads metadata and abstracts of arXiv papers with at least one of the corpus categories
   (cs.AI, cs.LG, stat.ML, cs.CL, cs.CV, cs.RO, cs.NE; primary or cross-listed) through the official
   arXiv API, window by window;
2. cleans them (duplicates, withdrawn papers, very short abstracts) and writes the final CSV table;
3. runs quick sanity checks on the table.

The heavy lifting is in `src/arxiv_ai_dataset.py`. Downloads are **resumable**: if the notebook is
interrupted, run the download cell again with the same parameters and it continues from the last
finished window (raw responses are cached, nothing is requested twice).

No text pre-processing is done here (no lowercasing, stemming, ...): it belongs to the project pipeline.

## 1. Configuration

Parameters still to be decided are marked `TODO`.

In [ ]:
# --- Parameters ----------------------------------------------------------------------------
CONTACT = "your.email@university.edu"   # put your e-mail: arXiv asks for an identifiable User-Agent

# A paper is in the corpus if it has at least one of these categories (primary or cross-listed)
CORPUS_CATEGORIES = ["cs.AI", "cs.LG", "stat.ML", "cs.CL", "cs.CV", "cs.RO", "cs.NE"]

START_DATE = "2015-01-01"      # TODO: first day to download (YYYY-MM-DD)
END_DATE = "2026-10-15"        # TODO: last day (inclusive); keep a few weeks before today,
                               #       recent papers can still be re-classified by moderators
WINDOW_DAYS = 7                # window length in days. With NO cap the window size does not change WHAT is
                               # downloaded (all papers), only how many requests are needed: 7 is much faster than 1
MAX_PER_WINDOW = 100_000       # no sampling cap: everything is downloaded (cut/sample later, in pandas)
SAMPLING = "first"             # irrelevant without a cap ("first" saves one request per window)
PAGE_SIZE = 500                # papers per HTTP request (the API allows up to 2000)
MIN_WORDS = 30                 # abstracts shorter than this are dropped
SEED = 42

OUTPUT_CSV = "../data/arxiv_ai_abstracts.csv"
CACHE_DIR = "../data/arxiv_cache"
CHECKPOINT_EVERY = 100         # also save a partial CSV every N finished windows (0 = off)
EXCEL_FRIENDLY = False         # True: ';'-separated CSV with BOM (double-click in Excel with Italian settings)
EXTRA_ARGS = []                # extra command-line options of the script, if needed

### Optional: run on Google Colab (data saved on Google Drive)

The download needs **no GPU** and no heavy computation, only time. If you run it on Colab, the cell below mounts
Google Drive and stores the cache and the CSV there, so everything survives a disconnection or a change of Google
account (share the Drive folder, or download the CSV). Every finished window is saved immediately, and a partial CSV
is refreshed every `CHECKPOINT_EVERY` windows. Outside Colab the cell does nothing. (Requires the repository to be pushed to GitHub.)

In [ ]:
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/Txt_lab"):
        subprocess.run(["git", "clone", "https://github.com/ilaria-cesa/Txt_lab", "/content/Txt_lab"], check=True)
    os.chdir("/content/Txt_lab/notebooks")
    DATA_DIR = "/content/drive/MyDrive/Txt_lab_data"
    os.makedirs(DATA_DIR, exist_ok=True)
    OUTPUT_CSV = f"{DATA_DIR}/arxiv_ai_abstracts.csv"
    CACHE_DIR = f"{DATA_DIR}/arxiv_cache"
    print("Colab: data will be saved in", DATA_DIR)

## 2. Download and build the table

This cell can run for hours on the full period. The progress bar shows the estimated remaining time; interrupt the kernel at any time and re-run to resume.

In [ ]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path("..") / "src"))
import arxiv_ai_dataset as ad

argv = [
    "--contact", CONTACT,
    "--start-date", START_DATE, "--end-date", END_DATE,
    "--window-days", str(WINDOW_DAYS), "--max-per-window", str(MAX_PER_WINDOW),
    "--corpus-categories", ",".join(CORPUS_CATEGORIES),
    "--sampling", SAMPLING, "--page-size", str(PAGE_SIZE), "--seed", str(SEED), "--min-words", str(MIN_WORDS),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--output", OUTPUT_CSV, "--cache-dir", CACHE_DIR,
] + (["--excel"] if EXCEL_FRIENDLY else []) + list(EXTRA_ARGS)

assert "@" in CONTACT and "your.email" not in CONTACT, "set CONTACT to your real e-mail (arXiv asks for it)"
return_code = ad.main(argv)
print("return code:", return_code, "(0 = ok, 1 = error, 130 = interrupted; re-run to resume)")

## 3. Sanity checks on the table

In [ ]:
import pandas as pd
sep = ";" if EXCEL_FRIENDLY else ","
df = pd.read_csv(OUTPUT_CSV, sep=sep, encoding="utf-8-sig" if EXCEL_FRIENDLY else "utf-8")
print(df.shape)
assert df["arxiv_id"].is_unique, "duplicated arxiv_id"
missing = df.isna().sum()[lambda s: s > 0]
print("missing values per column:", missing.to_dict() if len(missing) else "none")
df.head(3)

In [ ]:
# abstract length (in words) and date range
words = df["abstract"].str.split().str.len()
print(words.describe().round(1).to_string())
print("published:", df["published"].min(), "->", df["published"].max())
print("abstracts with a newline or a double space:", int(df["abstract"].str.contains(r"\n|  ").sum()))

### Papers per year and sampling coverage

Coverage = papers fetched / papers existing in the API for the same windows. Low coverage means that the sampling cap was hit: use **proportions within each year**, not raw counts, for trends.

In [ ]:
rows = []
for f in sorted(pathlib.Path(CACHE_DIR, "windows").glob("*.json")):
    w = json.loads(f.read_text(encoding="utf-8"))
    rows.append({"year": int(w["start"][:4]), "existing": w["total"], "fetched": w["n_fetched"]})
cov = pd.DataFrame(rows).groupby("year").sum()
cov["in_final_csv"] = df.groupby("year").size()
cov["coverage"] = (cov["fetched"] / cov["existing"]).round(3)
cov

### Labels

In [ ]:
top = df["primary_category"].value_counts()
print("distinct primary categories:", len(top))
print("generic AI labels (cs.AI / cs.LG / stat.ML as primary): %.1f%%" % (100 * df["primary_is_generic_ai"].mean()))
top.head(20).to_frame("n").join(df.drop_duplicates("primary_category").set_index("primary_category")["label_description"])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("year").size().plot.bar(ax=ax[0], title="Papers per year (final table)")
df["label_area"].value_counts().plot.barh(ax=ax[1], title="label_area").invert_yaxis()
plt.tight_layout(); plt.show()

## Notes

- `categories` and `n_categories` contain the label (the primary category is one of the categories):
  **never use them as features** for classification.
- Labels are chosen by the authors and only partly reviewed by moderators: they are noisy.
- Category descriptions come from `src/arxiv_taxonomy.py` (copied from the official taxonomy on 2026-11-13).
  Codes that are not in the table appear in the summary printed at the end of the download.